# 05. Deep Autoencoder - Training, Scoring & Validation Triage Evaluation

**Project:** Card Fraud Triage via Gaussian Mixture Density Scoring and Autoencoder Anomaly Detection  
**Phase:** Phase 2 — Autoencoder Anomaly Detection  
**Module:** Task 3B — Full Training, Reconstruction Error Scoring & Validation Benchmark  
**Audience:** Academic Review & College Viva Preparation  

---

## 1. Executive Summary & Objective

In this notebook, we build, train, and evaluate a **Deep Autoencoder** for credit card fraud triage.

### Why Autoencoders for Anomaly Detection?
1. **Unsupervised Normalcy Learning:** We train the autoencoder strictly on **legitimate transactions** ($y_{\text{train}} == 0$). It learns how normal cardholder transactions correlate across 29 standardized features.
2. **Information Bottleneck:** By compressing 29 features down to a 5-dimensional bottleneck and then expanding them back to 29, the network learns an efficient, low-dimensional representation of legitimate spending habits.
3. **Reconstruction Error as Anomaly Score:**
   $$\text{Anomaly Score}(x) = \text{MSE}(x, \hat{x}) = \frac{1}{29} \sum_{d=1}^{29} (x_d - \hat{x}_d)^2$$
   When an unseen fraud transaction passes through, the network cannot reconstruct its abnormal patterns accurately, producing a **high reconstruction error** that flags it for analyst review.
4. **Leak-Free Protocol:** Zero fraudulent transactions are ever shown during model training or early stopping, and the test set remains strictly untouched.

## 2. Environment Setup & Dependency Verification

We import PyTorch, NumPy, Pandas, Scikit-learn, and Matplotlib. To guarantee 100% reproducible results during viva presentation, random seeds are pinned and computations run on CPU.

In [2]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, precision_recall_curve

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.autoencoder_model import (
    Autoencoder,
    DEFAULT_AE_CONFIG,
    fit_autoencoder,
    create_autoencoder_model,
    save_autoencoder_model,
    set_seed
)
from src.autoencoder_scoring import compute_autoencoder_scores
from src.scoring import create_score_dataframe, save_scores, summarize_scores
from src.triage_evaluation import (
    load_validation_data,
    align_scores_with_labels,
    recall_at_fpr
)

# Set random seeds for exact reproducibility
set_seed(42)

print("Environment Configuration:")
print(f"  PyTorch version:    {torch.__version__}")
print(f"  Project Root:       {PROJECT_ROOT}")
print(f"  Compute Device:     CPU (Pinned for determinism)")

Environment Configuration:
  PyTorch version:    2.14.1
  Project Root:       /Users/kaustubhhiwanj/card-fraud-triage
  Compute Device:     CPU (Pinned for determinism)


## 3. Load Preprocessed Data & Verify Strict Data Hygiene

We load preprocessed features from `data/processed/`.
Notice the strict leak-free rules enforced here:
* **Legitimate Training Only:** Only $y_{\text{train}} == 0$ rows are fed to the optimizer. All 360 fraud cases in the training window are filtered out.
* **Validation Monitoring Only:** Early stopping monitors legitimate validation transactions ($y_{\text{val}} == 0$).
* **Test Isolation:** Test features (`X_test`) are never loaded or referenced.

In [3]:
data_dir = os.path.join(PROJECT_ROOT, "data", "processed")

# Load training data
X_train = np.load(os.path.join(data_dir, "X_train.npy"))
y_train = np.load(os.path.join(data_dir, "y_train.npy"))
row_ids_train = np.load(os.path.join(data_dir, "row_ids_train.npy"))

# Load validation data
val_dict = load_validation_data(data_dir)
X_val = val_dict["X_val"]
y_val = val_dict["y_val"]
row_ids_val = val_dict["row_ids_val"]

with open(os.path.join(data_dir, "feature_names.json"), "r") as f:
    feature_names = json.load(f)

print("Partition Dimensions:")
print(f"  X_train: {X_train.shape[0]:,} rows | Legit: {(y_train == 0).sum():,} | Fraud (Filtered out): {(y_train == 1).sum():,}")
print(f"  X_val:   {X_val.shape[0]:,} rows  | Legit: {(y_val == 0).sum():,} | Fraud (Evaluated):    {(y_val == 1).sum():,}")
print(f"  Features ({len(feature_names)}): {feature_names[:3]} ... {feature_names[-1]}")

Partition Dimensions:
  X_train: 170,888 rows | Legit: 170,528 | Fraud (Filtered out): 360
  X_val:   56,957 rows  | Legit: 56,900 | Fraud (Evaluated):    57
  Features (29): ['V1', 'V2', 'V3'] ... Amount


## 4. Autoencoder Architecture Inspection

The network is a symmetric fully connected multi-layer perceptron (MLP):
* **Input Layer:** 29 standardized features ($V_1$ to $V_{28}$ + $\ln(1 + \text{Amount})$).
* **Encoder:** $29 \to 20 \to 10 \to 5$ (reduces dimensionality by $5.8\times$ to a 5D bottleneck).
* **Decoder:** $5 \to 10 \to 20 \to 29$ (reconstructs standardized features).
* **Activations:** ReLU on hidden layers; **Linear** output (because standardized inputs have unbounded positive and negative values).

In [4]:
sample_model = create_autoencoder_model({"input_dim": 29, "bottleneck_dim": 5})
print(sample_model)

total_params = sum(p.numel() for p in sample_model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,}")

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=29, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=5, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=5, out_features=10, bias=True)
    (1): ReLU()
    (2): Linear(in_features=10, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=29, bias=True)
  )
)

Total Trainable Parameters: 1,754
